In [ ]:
import pandas as pd
import re
from pathlib import Path

In [9]:
# Load dataset as strings; don't auto-convert blanks to NaN
df = pd.read_csv("triplets_unified.csv", dtype=str, keep_default_na=False)

# Keeping only essential fields for graph
df = df.dropna(subset=["head", "relation", "tail"])

print(f"Records after cleaning: {df.shape[0]}")
df.head()

Records after cleaning: 633170


,head,relation,tail,source,pubmed_ids,chemical_id,disease_id
0,10074-G5,associated_with,Adenocarcinoma,chem_dis,26432044,C534883,MESH:D000230
1,10074-G5,associated_with,Adenocarcinoma of Lung,chem_dis,26656844|27602772,C534883,MESH:D000077192
2,10074-G5,associated_with,Alopecia,chem_dis,15902657,C534883,MESH:D000505
3,10074-G5,associated_with,Androgen-Insensitivity Syndrome,chem_dis,1303262|8281139,C534883,MESH:D013734
4,10074-G5,associated_with,Astrocytoma,chem_dis,24680642,C534883,MESH:D001254


In [10]:
# RENAME: head->Subject, relation->Predicate, tail->Object
df = df.rename(columns={
    "head": "Subject",
    "relation": "Predicate",
    "tail": "Object"
})

df.head()

,Subject,Predicate,Object,source,pubmed_ids,chemical_id,disease_id
0,10074-G5,associated_with,Adenocarcinoma,chem_dis,26432044,C534883,MESH:D000230
1,10074-G5,associated_with,Adenocarcinoma of Lung,chem_dis,26656844|27602772,C534883,MESH:D000077192
2,10074-G5,associated_with,Alopecia,chem_dis,15902657,C534883,MESH:D000505
3,10074-G5,associated_with,Androgen-Insensitivity Syndrome,chem_dis,1303262|8281139,C534883,MESH:D013734
4,10074-G5,associated_with,Astrocytoma,chem_dis,24680642,C534883,MESH:D001254


In [11]:
# PREDICATE NORMALIZATION

PREDICATE_MAP = {
    "affects": "AFFECTS",
    "associated_with": "ASSOCIATED_WITH",
    "linked_to": "LINKED_TO",    
    "increases": "INCREASES",
    "decreases": "DECREASES",
    "marker_mechanism": "MARKER_MECHANISM",
    "therapeutic": "TREATS",
}

def normalize_predicate(x: str) -> str | None:
    if x is None:
        return None
    base = str(x).strip().lower()
    # minimal cleanup → underscores, strip, etc.
    base = re.sub(r"\s+", "_", base)      # spaces -> _
    base = re.sub(r"[^\w]+", "_", base)   # punctuation -> _
    base = base.strip("_")
    return PREDICATE_MAP.get(base, base.upper()) if base else None

df["Predicate"] = df["Predicate"].apply(normalize_predicate)

print(df["Predicate"].value_counts())


Predicate
ASSOCIATED_WITH     464608
DECREASES            60946
AFFECTS              53047
INCREASES            42632
LINKED_TO             5820
MARKER_MECHANISM      3619
TREATS                2498
Name: count, dtype: int64


In [ ]:
# LIGHTWEIGHT LABEL CLEANUP on Subject/Object
# Keep acronyms/IDs (2+ uppercase) or hyphenated/digit words as-is in title-casing
def safe_title_case(text: str) -> str:
    out = []
    for w in str(text).split():
        if re.match(r"^[A-Z]{2,}$", w) or re.search(r"[\d-]", w):
            out.append(w)                 # preserve acronyms / IDs / hyphenated forms
        else:
            out.append(w.capitalize())
    return " ".join(out)

# (A) Swap only "X, Y" where BOTH sides are strictly alphabetic words (no digits/hyphens/parens)
ALPHA_ONLY = re.compile(r"^[A-Za-z ]+$")
def maybe_swap_two_part_comma(label: str) -> str:
    if "," not in label:
        return label
    parts = [p.strip() for p in label.split(",")]
    if len(parts) == 2 and all(parts) and ALPHA_ONLY.match(parts[0]) and ALPHA_ONLY.match(parts[1]):
        return f"{parts[1]} {parts[0]}"
    return label

# (B) if it looks like an IUPAC-like chemical it will do not touch it
IUPAC_PATTERN = re.compile(r"""
    \d+,\d+                |  
    \dH                    |  
    \d                     |  
    (?:^|[^\w])(N|O|S|R)-  |
    (?:cis|trans|iso|tert|sec|neo)- |
    \([^)]+\)              |  # parentheses content
    (?:[A-Za-z0-9]+-){2,}[A-Za-z0-9]+  # many hyphens in a single token
""", re.IGNORECASE | re.VERBOSE)

def looks_iupac(s: str) -> bool:
    return bool(IUPAC_PATTERN.search(str(s)))

def refine_label(label: str) -> str | None:
    if pd.isna(label):
        return None
    x = str(label).replace("_", " ").strip()

    # not touching IUPAC-like strings (chemicals stay exactly as-is)
    if looks_iupac(x):
        return x

    # For non-chemicals: safe reorder + tidy spaces + acronym-aware title
    x = maybe_swap_two_part_comma(x)
    x = re.sub(r"\s+", " ", x).strip()
    x = safe_title_case(x)
    return x or None

# Applying only to Subject/Object
df["Subject"] = df["Subject"].apply(refine_label)
df["Object"]  = df["Object"].apply(refine_label)

df.head()


,Subject,Predicate,Object,source,pubmed_ids,chemical_id,disease_id
0,10074-G5,ASSOCIATED_WITH,Adenocarcinoma,chem_dis,26432044,C534883,MESH:D000230
1,10074-G5,ASSOCIATED_WITH,Adenocarcinoma Of Lung,chem_dis,26656844|27602772,C534883,MESH:D000077192
2,10074-G5,ASSOCIATED_WITH,Alopecia,chem_dis,15902657,C534883,MESH:D000505
3,10074-G5,ASSOCIATED_WITH,Androgen-Insensitivity Syndrome,chem_dis,1303262|8281139,C534883,MESH:D013734
4,10074-G5,ASSOCIATED_WITH,Astrocytoma,chem_dis,24680642,C534883,MESH:D001254


In [13]:
# CASE-INSENSITIVE SYNONYM MAPPING on Subject/Object
synonym_map = {
    "hair loss": "Alopecia",
    "high blood sugar": "Hyperglycemia",
    "male breast cancer": "Breast Cancer, Male",
    # extend as needed; keys are lowercase; values are exactly as you want them to appear
}

def map_synonyms(label: str) -> str:
    if label is None:
        return None
    return synonym_map.get(str(label).lower(), label)

df["Subject"] = df["Subject"].apply(map_synonyms)
df["Object"]  = df["Object"].apply(map_synonyms)

changed = (df["Subject"].isin(synonym_map.values()) | df["Object"].isin(synonym_map.values()))
print(f"Rows affected by synonym mapping: {changed.sum():,}")
df[changed].head()


Rows affected by synonym mapping: 1,250


,Subject,Predicate,Object,source,pubmed_ids,chemical_id,disease_id
2,10074-G5,ASSOCIATED_WITH,Alopecia,chem_dis,15902657,C534883,MESH:D000505
190,"10,11-dihydro-5H-dibenzo(a,d)cycloheptene",ASSOCIATED_WITH,Alopecia,chem_dis,15902657,C515697,MESH:D000505
1323,"10-methoxy-2,2-dimethyl-2,6-dihydropyrano(3,2-...",ASSOCIATED_WITH,Alopecia,chem_dis,20561897,C554291,MESH:D000505
1776,10-nitro-oleic acid,ASSOCIATED_WITH,Alopecia,chem_dis,20561897,C521487,MESH:D000505
1983,10-nitro-oleic acid,ASSOCIATED_WITH,Hyperglycemia,chem_dis,14514642,C521487,MESH:D006943


In [15]:
# FINAL CLEANUP, saving S/P/O columns
# triming whitespace on core fields
for col in ["Subject", "Predicate", "Object"]:
    df[col] = df[col].astype(str).str.strip()

# droping incomplete rows - if core fields are empty
df = df.replace({"": None})
df = df.dropna(subset=["Subject", "Predicate", "Object"])

# saving
output_path = "unified_triplets_normalized.csv"
df.to_csv(output_path, index=False)
print(f"Saved cleaned dataset -> {output_path} (rows: {len(df):,})")


Saved cleaned dataset -> unified_triplets_normalized.csv (rows: 633,170)


# *Neo4J Part*

In [17]:
# load & verify input

IN_PATH = "unified_triplets_normalized.csv"
OUT_DIR = Path("neo4j_export"); OUT_DIR.mkdir(exist_ok=True)

REQUIRED_COLS = ["Subject", "Predicate", "Object", "source", "pubmed_ids", "chemical_id", "disease_id"]
APPROVED_PREDICATES = {"AFFECTS","ASSOCIATED_WITH","LINKED_TO","INCREASES","DECREASES","MARKER_MECHANISM","TREATS"}

df = pd.read_csv(IN_PATH, dtype=str, keep_default_na=False)

missing = [c for c in REQUIRED_COLS if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

print(f"Input rows: {len(df):,}")
print("Predicates (top 10):")
print(df["Predicate"].value_counts().head(10))

# # Optional safety: ensure predicate vocabulary is as expected
# unknown = set(df["Predicate"].unique()) - APPROVED_PREDICATES
# if unknown:
#     raise ValueError(f"Unexpected predicates present: {sorted(unknown)}")


Input rows: 633,170
Predicates (top 10):
Predicate
ASSOCIATED_WITH     464608
DECREASES            60946
AFFECTS              53047
INCREASES            42632
LINKED_TO             5820
MARKER_MECHANISM      3619
TREATS                2498
Name: count, dtype: int64


In [18]:
# chemicals node file
chem = (
    df.loc[df["chemical_id"].astype(str).str.strip() != "", ["chemical_id", "Subject"]]
      .drop_duplicates(subset=["chemical_id"])
      .rename(columns={"Subject": "name"})
)

chem_path = OUT_DIR / "chemicals.csv"
chem.to_csv(chem_path, index=False)
print(f"chemicals.csv -> {chem_path} | rows: {len(chem):,}")


chemicals.csv -> neo4j_export\chemicals.csv | rows: 3,688


In [19]:
# Diseases node file
dis = (
    df.loc[df["disease_id"].astype(str).str.strip() != "", ["disease_id", "Object"]]
      .drop_duplicates(subset=["disease_id"])
      .rename(columns={"Object": "name"})
)

dis_path = OUT_DIR / "diseases.csv"
dis.to_csv(dis_path, index=False)
print(f"diseases.csv -> {dis_path} | rows: {len(dis):,}")


diseases.csv -> neo4j_export\diseases.csv | rows: 5,930


In [20]:
# Relationships file (one row per input row; provenance-preserving)
# Keeping only rows that have both IDs (joinable). not modifying source/pubmed_ids.
edges_cols = ["chemical_id", "disease_id", "Predicate", "source", "pubmed_ids"]
edges = (
    df.loc[
        (df["chemical_id"].astype(str).str.strip() != "") &
        (df["disease_id"].astype(str).str.strip() != "")
    , edges_cols].copy()
)

edges_path = OUT_DIR / "chem_dis_edges.csv"
edges.to_csv(edges_path, index=False)
print(f"chem_dis_edges.csv -> {edges_path} | rows: {len(edges):,}")


chem_dis_edges.csv -> neo4j_export\chem_dis_edges.csv | rows: 470,725


In [21]:
# quick validation of staging files
print("\nStaging summary")
print("Chemicals:", len(chem))
print("Diseases :", len(dis))
print("Edges    :", len(edges))

# Basic joinability check (sample): all IDs in edges exist in node files
chem_ids = set(chem["chemical_id"])
dis_ids  = set(dis["disease_id"])

missing_chem = (~edges["chemical_id"].isin(chem_ids)).sum()
missing_dis  = (~edges["disease_id"].isin(dis_ids)).sum()

print(f"Edges with missing chemical_id in chemicals.csv: {missing_chem}")
print(f"Edges with missing disease_id in diseases.csv : {missing_dis}")

# Relationship vocabulary check (again on the staged edges)
print("\nRelationship types (edges):")
print(edges["Predicate"].value_counts())



Staging summary
Chemicals: 3688
Diseases : 5930
Edges    : 470725
Edges with missing chemical_id in chemicals.csv: 0
Edges with missing disease_id in diseases.csv : 0

Relationship types (edges):
Predicate
ASSOCIATED_WITH     464608
MARKER_MECHANISM      3619
TREATS                2498
Name: count, dtype: int64
